In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from sklearn.cluster import BisectingKMeans, AgglomerativeClustering
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import silhouette_score, pairwise_distances
import numpy as np
from scipy.cluster.hierarchy import dendrogram, linkage, cut_tree

## 1: Ler dados

In [12]:
df_species = pd.read_csv("./dataset/species_dataset.csv")
df_species.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   species_id          600 non-null    str    
 1   body_mass_kg        600 non-null    float64
 2   num_legs            600 non-null    int64  
 3   has_wings           600 non-null    int64  
 4   tail_length_cm      600 non-null    float64
 5   diet_type           600 non-null    str    
 6   eye_count           600 non-null    int64  
 7   skin_type           600 non-null    str    
 8   nocturnal           600 non-null    int64  
 9   avg_lifespan_years  600 non-null    float64
 10  has_venom           600 non-null    int64  
 11  social_behavior     600 non-null    str    
dtypes: float64(3), int64(5), str(4)
memory usage: 56.4 KB


## 2: Filtrar Dados

Resultado: Não há dados faltantes, mas a coluna de ID é inútil para o agrupamento, já que não é um campo em comum entre quaisquer 2 registros. Logo, ela foi retirada.

In [13]:
df_species.drop(columns=['species_id'], inplace=True)
df_species.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   body_mass_kg        600 non-null    float64
 1   num_legs            600 non-null    int64  
 2   has_wings           600 non-null    int64  
 3   tail_length_cm      600 non-null    float64
 4   diet_type           600 non-null    str    
 5   eye_count           600 non-null    int64  
 6   skin_type           600 non-null    str    
 7   nocturnal           600 non-null    int64  
 8   avg_lifespan_years  600 non-null    float64
 9   has_venom           600 non-null    int64  
 10  social_behavior     600 non-null    str    
dtypes: float64(3), int64(5), str(3)
memory usage: 51.7 KB


In [18]:
df_species.describe()

,body_mass_kg,num_legs,has_wings,tail_length_cm,eye_count,nocturnal,avg_lifespan_years,has_venom
count,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000,600.000000
mean,49.784101,3.950000,0.285000,45.052394,2.203333,0.380000,120.024664,0.150000
std,19.297540,2.833882,0.451791,9.930405,1.074203,0.485791,0.976676,0.357369
min,5.000000,0.000000,0.000000,16.287173,0.000000,0.000000,117.364032,0.000000
25%,36.107729,2.000000,0.000000,38.446490,2.000000,0.000000,119.366525,0.000000
50%,50.154768,4.000000,0.000000,45.266501,2.000000,0.000000,120.008112,0.000000
75%,62.463348,6.000000,1.000000,51.858016,2.000000,1.000000,120.684532,0.000000
max,127.054630,8.000000,1.000000,72.308672,4.000000,1.000000,123.657702,1.000000


In [15]:
df_species['social_behavior'].unique()

<StringArray>
['pair-living', 'solitary', 'group-living']
Length: 3, dtype: str

In [16]:
df_species['diet_type'].unique()

<StringArray>
['carnivore', 'herbivore', 'omnivore']
Length: 3, dtype: str

In [17]:
df_species['skin_type'].unique()

<StringArray>
['fur', 'scales', 'feathers', 'skin']
Length: 4, dtype: str

## 3: Análise Exploratória